# LeetCode 862 — Shortest Subarray with Sum at Least K bằng Python

- Link: https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/
- Độ khó: **Hard**
- Chủ đề: Tổng tiền tố, Hàng đợi hai đầu đơn điệu (monotonic deque), Heap, Ngăn xếp đơn điệu + tìm kiếm nhị phân

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def shortestSubarray(self, nums: List[int], k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Tìm độ dài **ngắn nhất** của một mảng con **liên tiếp, khác rỗng** có tổng **ít nhất** `k`. Không có thì trả `-1`.

Nếu mọi phần tử đều dương, đây là bài cửa sổ trượt quen thuộc (LeetCode 209). Cái khó ở đây: `nums` có thể chứa **số âm**, làm cửa sổ trượt mất tính đơn điệu — mở rộng cửa sổ có thể làm tổng **giảm**. Lời giải chuẩn chuyển sang **tổng tiền tố** `P` và dùng **hàng đợi hai đầu đơn điệu tăng** chứa các chỉ số tiền tố "còn hữu ích", đạt `O(n)`.

Sau notebook này bạn sẽ:

1. Viết lại tổng mảng con bằng hiệu hai tổng tiền tố: `sum(nums[i:j]) = P[j] − P[i]`.
2. Hiểu vì sao cửa sổ trượt **sai** khi có số âm.
3. Nắm hai quy tắc loại bỏ của deque: **bỏ đầu** khi đã dùng xong, **bỏ đuôi** khi bị "lấn át".
4. Cài thêm hai lời giải `O(n log n)`: heap theo tiền tố và ngăn xếp đơn điệu + tìm kiếm nhị phân.
5. Kiểm thử kỹ với dữ liệu có số âm.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên `nums` và số nguyên `k`. Trả về độ dài nhỏ nhất của một mảng con khác rỗng (các phần tử liên tiếp) có tổng `>= k`. Nếu không tồn tại, trả `-1`.

**Ví dụ 1**

```text
Input: nums = [1], k = 1
Output: 1
```

**Ví dụ 2**

```text
Input: nums = [1,2], k = 4
Output: -1
```

**Ví dụ 3**

```text
Input: nums = [2,-1,2], k = 3
Output: 3
```

**Ràng buộc**

- `1 <= nums.length <= 10⁵`
- `−10⁵ <= nums[i] <= 10⁵`
- `1 <= k <= 10⁹`

In [1]:
from typing import List

EXAMPLES = [
    ([1], 1, 1),
    ([1, 2], 4, -1),
    ([2, -1, 2], 3, 3),
    ([84, -37, 32, 40, 95], 167, 3),     # [32, 40, 95]
    ([-28, 81, -20, 28, -29], 89, 3),    # [81, -20, 28]
    ([1, -1, 1, -1, 5], 5, 1),
]
for a, k, exp in EXAMPLES:
    print(a, "k =", k, "-> kỳ vọng", exp)

[1] k = 1 -> kỳ vọng 1
[1, 2] k = 4 -> kỳ vọng -1
[2, -1, 2] k = 3 -> kỳ vọng 3
[84, -37, 32, 40, 95] k = 167 -> kỳ vọng 3
[-28, 81, -20, 28, -29] k = 89 -> kỳ vọng 3
[1, -1, 1, -1, 5] k = 5 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Tổng tiền tố

`P[0] = 0`, `P[j] = nums[0] + ... + nums[j−1]`. Tổng đoạn `nums[i..j−1]` (độ dài `j − i`) là `P[j] − P[i]`.

In [2]:
from itertools import accumulate

nums = [2, -1, 2]
P = [0] + list(accumulate(nums))
print(P)
print("tổng nums[0:3] =", P[3] - P[0], "| tổng nums[1:3] =", P[3] - P[1])

[0, 2, 1, 3]
tổng nums[0:3] = 3 | tổng nums[1:3] = 1


### 3.2. `collections.deque` — thao tác hai đầu `O(1)`

`append`, `pop` (đuôi), `popleft` (đầu), `dq[0]` (đầu), `dq[-1]` (đuôi).

In [3]:
from collections import deque
dq = deque([1, 2, 3])
dq.append(4); dq.popleft(); dq.pop()
print(dq, dq[0], dq[-1])

deque([2, 3]) 2 3


### 3.3. Hàng đợi đơn điệu (monotonic queue)

Giữ deque sao cho giá trị (theo một khoá) **tăng dần** từ đầu tới đuôi: trước khi thêm phần tử mới, bỏ khỏi đuôi mọi phần tử có khoá ≥ khoá mới.

In [4]:
def push_increasing(dq, vals, idx):
    while dq and vals[dq[-1]] >= vals[idx]:
        dq.pop()
    dq.append(idx)

vals = [3, 1, 4, 1, 5]
dq = deque()
for i in range(len(vals)):
    push_increasing(dq, vals, i)
    print(i, [vals[j] for j in dq])

0 [3]
1 [1]
2 [1, 4]
3 [1]
4 [1, 5]


### 3.4. Tìm kiếm nhị phân với `bisect`

`bisect_right(a, x)` = số phần tử `<= x` trong mảng tăng `a`.

In [5]:
from bisect import bisect_right
a = [0, 2, 5, 9]
print(bisect_right(a, 5), bisect_right(a, 4))

3 2


### 3.5. Heap với bộ `(giá trị, chỉ số)`

`heapq` so sánh tuple theo phần tử đầu, hoà thì phần tử sau.

In [6]:
import heapq
h = []
for i, v in enumerate([5, 1, 3]):
    heapq.heappush(h, (v, i))
print(heapq.heappop(h), h[0])

(1, 1) (3, 2)


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Phát biểu lại bằng tiền tố

Tìm cặp `i < j` sao cho `P[j] − P[i] >= k` và `j − i` nhỏ nhất.

Với ví dụ 3: `P = [0, 2, 1, 3]`, `k = 3`.

| j | P[j] | các i với P[j] − P[i] ≥ 3 | độ dài ngắn nhất |
|---|---|---|---|
| 1 | 2 | — | — |
| 2 | 1 | — | — |
| 3 | 3 | i = 0 (3 − 0 = 3) | 3 |

→ Đáp án 3.

### 4.2. Hai quan sát vàng

Cố định `j`, ta muốn `i` **lớn nhất** (gần `j` nhất) có `P[i] <= P[j] − k`.

1. **Lấn át (bỏ đuôi):** nếu `i₁ < i₂` và `P[i₁] >= P[i₂]`, thì `i₁` **không bao giờ** tốt hơn `i₂` cho mọi `j` sau: `i₂` gần hơn và cho tổng `P[j] − P[i₂] >= P[j] − P[i₁]`. → Bỏ `i₁`. Các ứng viên còn lại có `P` **tăng dần** theo chỉ số.
2. **Dùng xong (bỏ đầu):** nếu `P[j] − P[i] >= k` với `i` ở đầu deque, ghi nhận `j − i` rồi **bỏ `i`**: với `j' > j` bất kỳ, độ dài `j' − i > j − i` — không thể tốt hơn.

### 4.3. Vì sao cửa sổ trượt sai?

Với số dương, khi tổng đủ thì thu hẹp trái là an toàn. Với số âm: `[84, -37, 32, 40, 95]`, `k = 167`. Cửa sổ trượt kiểu 209 sẽ bỏ qua đáp án — xem lỗi 1 ở mục 13.

In [7]:
nums = [84, -37, 32, 40, 95]
P = [0] + list(accumulate(nums))
print("P =", P)
best = min((j - i, i, j) for j in range(len(P)) for i in range(j) if P[j] - P[i] >= 167)
print("cặp tốt nhất (độ dài, i, j):", best, "→ mảng con", nums[best[1]:best[2]])

P = [0, 84, 47, 79, 119, 214]
cặp tốt nhất (độ dài, i, j): (3, 2, 5) → mảng con [32, 40, 95]


## 5. Cách 1 — Brute Force: mọi cặp `(i, j)` với tổng tiền tố

**Ý tưởng.** Thử mọi điểm đầu `i`, cộng dần tới `j`; khi tổng ≥ `k` lần đầu thì ghi nhận độ dài và dừng (với `i` cố định, `j` nhỏ nhất là tốt nhất).

In [8]:
def shortest_brute(nums, k):
    n = len(nums)
    best = n + 1
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            if s >= k:
                best = min(best, j - i + 1)
                break
    return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert shortest_brute(a, k) == exp
print("Brute force: OK")

Brute force: OK


**Độ phức tạp.** `O(n²)` thời gian — `10¹⁰` với `n = 10⁵`. `O(1)` bộ nhớ.

**Nút thắt.** Với mỗi `j`, ta thử lại **mọi** `i`. Nhưng nhiều `i` bị lấn át (có `i'` gần hơn với `P[i'] <= P[i]`) hoặc đã dùng xong. Chỉ cần giữ tập ứng viên "hữu ích" — đó là deque đơn điệu.

## 6. Bước đột phá

**Insight 1 — Ứng viên hữu ích có `P` tăng dần.** Khi thêm chỉ số `j` vào tập ứng viên điểm đầu, mọi `i` ở đuôi với `P[i] >= P[j]` bị lấn át → bỏ. Deque giữ chỉ số tăng dần, `P` tăng dần.

**Insight 2 — Ghi nhận và bỏ ở đầu.** Với `j` hiện tại, chừng nào `P[j] − P[dq[0]] >= k`: ghi nhận `j − dq[0]` và `popleft`. Vì `P` tăng dần trong deque, các phần tử đầu là những phần tử có `P` nhỏ nhất — nếu đầu không thoả thì không phần tử nào thoả.

**Insight 3 — Mỗi chỉ số vào/ra deque tối đa một lần** → tổng `O(n)`.

**Thứ tự trong vòng lặp cho mỗi `j`:** (1) bỏ đầu & ghi nhận, (2) bỏ đuôi bị lấn át, (3) thêm `j`.

**Quy tắc nếu…thì:**
- *Nếu* `P[j] − P[dq[0]] >= k` *thì* cập nhật đáp án và `popleft`.
- *Nếu* `P[dq[-1]] >= P[j]` *thì* `pop` đuôi.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Heap các tiền tố (min-heap theo `P`), `O(n log n)`

Với mỗi `j`, các `i` có `P[i]` nhỏ nhất nằm ở đỉnh heap. Chừng nào `P[j] − đỉnh >= k`, ghi nhận `j − i` và **bỏ** đỉnh (dùng xong, như quan sát 2). Phần lấn át không cần xử lý tường minh.

In [9]:
def shortest_heap(nums, k):
    n = len(nums)
    best = n + 1
    heap = [(0, 0)]                  # (P[i], i)
    s = 0
    for j in range(1, n + 1):
        s += nums[j - 1]
        while heap and s - heap[0][0] >= k:
            best = min(best, j - heapq.heappop(heap)[1])
        heapq.heappush(heap, (s, j))
    return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert shortest_heap(a, k) == exp
print("Heap tiền tố: OK")

Heap tiền tố: OK


Vì sao bỏ đỉnh là an toàn? Đỉnh `(P[i], i)` thoả với `j` → mọi `j' > j` cho độ dài lớn hơn với cùng `i`. Ngoài ra các phần tử bị bỏ có `P` nhỏ nhất — phần tử còn lại có `P` lớn hơn, chưa bị ảnh hưởng.

Code nộp LeetCode cho cách 2:

In [10]:
class SolutionHeap:
    def shortestSubarray(self, nums: List[int], k: int) -> int:
        n = len(nums)
        best = n + 1
        heap = [(0, 0)]
        s = 0
        for j in range(1, n + 1):
            s += nums[j - 1]
            while heap and s - heap[0][0] >= k:
                best = min(best, j - heapq.heappop(heap)[1])
            heapq.heappush(heap, (s, j))
        return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert SolutionHeap().shortestSubarray(a, k) == exp
print("SolutionHeap: OK")

SolutionHeap: OK


**Độ phức tạp cách 2:** mỗi tiền tố vào/ra heap một lần → `O(n log n)`; bộ nhớ `O(n)`.

### 7.2. Cách 3 — Ngăn xếp đơn điệu + tìm kiếm nhị phân, `O(n log n)`

Giữ ngăn xếp các chỉ số có `P` tăng dần (chỉ áp dụng quy tắc "bỏ đuôi"). Với mỗi `j`, tìm nhị phân chỉ số `i` **lớn nhất** trong ngăn xếp có `P[i] <= P[j] − k`. Không bỏ đầu — nên phải tìm nhị phân thay vì nhìn đầu deque.

In [11]:
def shortest_stack_bisect(nums, k):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    st_idx, st_val = [], []          # chỉ số và P tương ứng, P tăng dần
    best = n + 1
    for j in range(n + 1):
        pos = bisect_right(st_val, P[j] - k)      # số phần tử có P <= P[j] - k
        if pos:
            best = min(best, j - st_idx[pos - 1])
        while st_val and st_val[-1] >= P[j]:
            st_val.pop(); st_idx.pop()
        st_idx.append(j); st_val.append(P[j])
    return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert shortest_stack_bisect(a, k) == exp
print("Ngăn xếp đơn điệu + nhị phân: OK")

Ngăn xếp đơn điệu + nhị phân: OK


Giải thích: trong ngăn xếp `P` tăng dần theo chỉ số tăng, nên phần tử có `P <= P[j] − k` tạo thành một **tiền tố** của ngăn xếp; phần tử cuối của tiền tố đó có chỉ số lớn nhất → độ dài nhỏ nhất.

Code nộp LeetCode cho cách 3:

In [12]:
class SolutionStack:
    def shortestSubarray(self, nums: List[int], k: int) -> int:
        n = len(nums)
        st_idx, st_val = [0], [0]
        best = n + 1
        s = 0
        for j in range(1, n + 1):
            s += nums[j - 1]
            pos = bisect_right(st_val, s - k)
            if pos:
                best = min(best, j - st_idx[pos - 1])
            while st_val and st_val[-1] >= s:
                st_val.pop()
                st_idx.pop()
            st_idx.append(j)
            st_val.append(s)
        return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert SolutionStack().shortestSubarray(a, k) == exp
print("SolutionStack: OK")

SolutionStack: OK


**Độ phức tạp cách 3:** `O(n log n)` (mỗi bước một lần tìm nhị phân), `O(n)` bộ nhớ.

### 7.3. Cách 4 — Hàng đợi hai đầu đơn điệu, `O(n)`

Kết hợp **cả hai** quy tắc: bỏ đuôi (lấn át) và bỏ đầu (dùng xong).

In [13]:
def shortest_deque(nums, k):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    dq = deque()
    best = n + 1
    for j in range(n + 1):
        while dq and P[j] - P[dq[0]] >= k:        # (1) dùng xong ở đầu
            best = min(best, j - dq.popleft())
        while dq and P[dq[-1]] >= P[j]:           # (2) bị lấn át ở đuôi
            dq.pop()
        dq.append(j)                              # (3) thêm ứng viên mới
    return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert shortest_deque(a, k) == exp
print("Deque đơn điệu: OK")

Deque đơn điệu: OK


Giải thích từng dòng:

- `P` có `n + 1` phần tử; chỉ số deque là chỉ số tiền tố (điểm đầu `i` của mảng con `nums[i:j]`).
- Vòng (1): đầu deque có `P` nhỏ nhất. Nếu `P[j] − P[đầu] >= k` thì mảng con `nums[đầu:j]` hợp lệ; ghi nhận và bỏ.
- Vòng (2): mọi đuôi có `P >= P[j]` bị `j` lấn át.
- `best = n + 1` làm "vô cực"; nếu không đổi thì trả `-1`.

Code nộp LeetCode cho cách 4:

In [14]:
class SolutionDeque:
    def shortestSubarray(self, nums: List[int], k: int) -> int:
        n = len(nums)
        P = [0] * (n + 1)
        for i, x in enumerate(nums):
            P[i + 1] = P[i] + x
        dq = deque()
        best = n + 1
        for j in range(n + 1):
            while dq and P[j] - P[dq[0]] >= k:
                best = min(best, j - dq.popleft())
            while dq and P[dq[-1]] >= P[j]:
                dq.pop()
            dq.append(j)
        return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert SolutionDeque().shortestSubarray(a, k) == exp
print("SolutionDeque: OK")

SolutionDeque: OK


**Độ phức tạp cách 4:** mỗi chỉ số vào và ra deque tối đa một lần → `O(n)`; bộ nhớ `O(n)`.

### 7.4. Cách 5 — Tìm kiếm nhị phân theo đáp án (độ dài `L`) + max tổng cửa sổ độ dài ≤ L

Một góc nhìn khác: tính chất "tồn tại mảng con độ dài ≤ L có tổng ≥ k" là **đơn điệu** theo `L`. Kiểm tra với `L` cố định: với mỗi `j`, cần `min P[i]` trên cửa sổ `i ∈ [j − L, j − 1]` — dùng deque min trượt `O(n)`. Tổng `O(n log n)`.

In [15]:
def exists_len_at_most(P, k, L):
    dq = deque()                      # chỉ số i, P[i] tăng dần (min ở đầu)
    for j in range(1, len(P)):
        i_new = j - 1
        while dq and P[dq[-1]] >= P[i_new]:
            dq.pop()
        dq.append(i_new)
        if dq[0] < j - L:
            dq.popleft()
        if P[j] - P[dq[0]] >= k:
            return True
    return False

def shortest_binary_search(nums, k):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    if not exists_len_at_most(P, k, n):
        return -1
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if exists_len_at_most(P, k, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for a, k, exp in EXAMPLES:
    assert shortest_binary_search(a, k) == exp
print("Nhị phân theo đáp án: OK")

Nhị phân theo đáp án: OK


Lưu ý: mỗi bước `j` chỉ thêm đúng một chỉ số `j − 1` nên phần tử quá hạn ở đầu tối đa một → `if` (không cần `while`) là đủ.

Code nộp LeetCode cho cách 5:

In [16]:
class SolutionBinary:
    def shortestSubarray(self, nums: List[int], k: int) -> int:
        return shortest_binary_search(nums, k)

for a, k, exp in EXAMPLES:
    assert SolutionBinary().shortestSubarray(a, k) == exp
print("SolutionBinary: OK")

SolutionBinary: OK


**Độ phức tạp cách 5:** `O(n log n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace cách 4 trên `nums = [84, -37, 32, 40, 95]`, `k = 167`.

In [17]:
def trace_deque(nums, k):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    print("P =", P)
    dq = deque()
    best = n + 1
    for j in range(n + 1):
        actions = []
        while dq and P[j] - P[dq[0]] >= k:
            i = dq.popleft()
            best = min(best, j - i)
            actions.append(f"bỏ đầu i={i} (P[j]-P[i]={P[j] - P[i]} ≥ {k}, độ dài {j - i})")
        while dq and P[dq[-1]] >= P[j]:
            i = dq.pop()
            actions.append(f"bỏ đuôi i={i} (P[i]={P[i]} ≥ P[j]={P[j]})")
        dq.append(j)
        print(f"j={j} P[j]={P[j]:>4} | {'; '.join(actions) or '-':60} | deque={list(dq)} P={[P[i] for i in dq]} | best={best if best <= n else '∞'}")
    return best if best <= n else -1

trace_deque([84, -37, 32, 40, 95], 167)

P = [0, 84, 47, 79, 119, 214]
j=0 P[j]=   0 | -                                                            | deque=[0] P=[0] | best=∞
j=1 P[j]=  84 | -                                                            | deque=[0, 1] P=[0, 84] | best=∞
j=2 P[j]=  47 | bỏ đuôi i=1 (P[i]=84 ≥ P[j]=47)                              | deque=[0, 2] P=[0, 47] | best=∞
j=3 P[j]=  79 | -                                                            | deque=[0, 2, 3] P=[0, 47, 79] | best=∞
j=4 P[j]= 119 | -                                                            | deque=[0, 2, 3, 4] P=[0, 47, 79, 119] | best=∞
j=5 P[j]= 214 | bỏ đầu i=0 (P[j]-P[i]=214 ≥ 167, độ dài 5); bỏ đầu i=2 (P[j]-P[i]=167 ≥ 167, độ dài 3) | deque=[3, 4, 5] P=[79, 119, 214] | best=3


3

Bảng trạng thái:

| j | P[j] | hành động | deque (chỉ số) | best |
|---|---|---|---|---|
| 0 | 0 | thêm 0 | [0] | ∞ |
| 1 | 84 | thêm 1 | [0, 1] | ∞ |
| 2 | 47 | bỏ đuôi 1 (84 ≥ 47) | [0, 2] | ∞ |
| 3 | 79 | thêm 3 | [0, 2, 3] | ∞ |
| 4 | 119 | thêm 4 | [0, 2, 3, 4] | ∞ |
| 5 | 214 | bỏ đầu 0 (214 ≥ 167, dài 5), bỏ đầu 2 (167 ≥ 167, dài 3) | [3, 4, 5] | 3 |

Đáp án 3 — mảng con `nums[2:5] = [32, 40, 95]`.

Trace thêm một ví dụ có đáp án ngắn nhờ "bỏ đuôi":

In [18]:
trace_deque([1, -1, 1, -1, 5], 5)

P = [0, 1, 0, 1, 0, 5]
j=0 P[j]=   0 | -                                                            | deque=[0] P=[0] | best=∞
j=1 P[j]=   1 | -                                                            | deque=[0, 1] P=[0, 1] | best=∞
j=2 P[j]=   0 | bỏ đuôi i=1 (P[i]=1 ≥ P[j]=0); bỏ đuôi i=0 (P[i]=0 ≥ P[j]=0) | deque=[2] P=[0] | best=∞
j=3 P[j]=   1 | -                                                            | deque=[2, 3] P=[0, 1] | best=∞
j=4 P[j]=   0 | bỏ đuôi i=3 (P[i]=1 ≥ P[j]=0); bỏ đuôi i=2 (P[i]=0 ≥ P[j]=0) | deque=[4] P=[0] | best=∞
j=5 P[j]=   5 | bỏ đầu i=4 (P[j]-P[i]=5 ≥ 5, độ dài 1)                       | deque=[5] P=[5] | best=1


1

## 9. Code nộp LeetCode

Chọn cách 4 (deque đơn điệu): `O(n)`.

In [19]:
from collections import deque
from typing import List

class Solution:
    def shortestSubarray(self, nums: List[int], k: int) -> int:
        n = len(nums)
        P = [0] * (n + 1)                      # tổng tiền tố
        for i, x in enumerate(nums):
            P[i + 1] = P[i] + x
        dq = deque()                           # chỉ số tiền tố, P tăng dần
        best = n + 1
        for j in range(n + 1):
            # (1) Đầu deque đã tạo được mảng con hợp lệ: ghi nhận rồi bỏ
            while dq and P[j] - P[dq[0]] >= k:
                best = min(best, j - dq.popleft())
            # (2) Bỏ các ứng viên ở đuôi bị j lấn át (P lớn hơn hoặc bằng)
            while dq and P[dq[-1]] >= P[j]:
                dq.pop()
            dq.append(j)
        return best if best <= n else -1

for a, k, exp in EXAMPLES:
    assert Solution().shortestSubarray(a, k) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

Gọi `ans` là độ dài tối ưu, đạt tại cặp `(i*, j*)` với `P[j*] − P[i*] >= k`, `j* − i*` nhỏ nhất. Ta chứng minh thuật toán ghi nhận một giá trị `<= j* − i*` (và mọi giá trị ghi nhận đều là độ dài của mảng con hợp lệ, nên `best >= ans`).

**Bất biến:** sau khi xử lý `j`, deque chứa các chỉ số tăng dần với `P` **tăng ngặt**.

**Xét `i*`.** Nó được thêm vào deque ở bước `j = i*`. Có hai khả năng trước khi tới bước `j*`:

1. `i*` bị **bỏ đầu** ở bước `j' <= j*`: khi đó đã ghi nhận `j' − i* <= j* − i*`. ✔
2. `i*` bị **bỏ đuôi** ở bước `j'` với `i* < j' <= j*` vì `P[i*] >= P[j']`. Nhưng khi đó `P[j*] − P[j'] >= P[j*] − P[i*] >= k` và `j* − j' < j* − i*` — mâu thuẫn với tính tối ưu, **trừ khi** `j' = j*`; mà ở bước `j*` vòng (1) chạy **trước** vòng (2), nên `i*` sẽ bị xét ở vòng (1) trước (nếu nó ở đầu) — xem điểm 3.
3. `i*` còn trong deque tới bước `j*`. Mọi phần tử đứng trước `i*` trong deque có `P < P[i*] <= P[j*] − k`, nên vòng (1) sẽ lần lượt bỏ chúng và ghi nhận, rồi tới `i*` — ghi nhận `j* − i*`. ✔

Vậy `best <= ans`, kết hợp `best >= ans` ⇒ `best = ans`. Nếu không có cặp nào, `best` giữ `n + 1` → trả `-1`. ∎

**Cách 2 (heap)** giống cách 4 nhưng không bỏ đuôi: phần tử bị lấn át vẫn nằm trong heap, không làm sai (chỉ thêm phép so sánh). **Cách 3** dùng tính tăng dần của `P` trong ngăn xếp để tìm nhị phân `i` lớn nhất thoả. **Cách 5** dựa trên tính đơn điệu của vị từ theo `L`.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. Heap tiền tố | `O(n log n)` | `O(n)` |
| 3. Ngăn xếp + nhị phân | `O(n log n)` | `O(n)` |
| 4. Deque đơn điệu | `O(n)` | `O(n)` |
| 5. Nhị phân theo đáp án | `O(n log n)` | `O(n)` |

In [20]:
import random, time

rnd = random.Random(862)
big = [rnd.randint(-10 ** 5, 10 ** 5) for _ in range(10 ** 5)]
for name, fn in [("heap", shortest_heap), ("stack", shortest_stack_bisect), ("deque", shortest_deque),
                 ("binary", shortest_binary_search), ("Solution", lambda a, k: Solution().shortestSubarray(a, k))]:
    t0 = time.perf_counter()
    res = (fn(big, 10 ** 6), fn(big, 10 ** 9))
    print(f"{name:9}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

heap     : (19, -1)  (90 ms)


stack    : (19, -1)  (63 ms)


deque    : (19, -1)  (46 ms)


binary   : (19, -1)  (117 ms)


Solution : (19, -1)  (56 ms)


## 12. Bộ test

In [21]:
methods = {
    "brute": shortest_brute,
    "heap": shortest_heap,
    "stack": shortest_stack_bisect,
    "deque": shortest_deque,
    "binary": shortest_binary_search,
    "SolutionHeap": lambda a, k: SolutionHeap().shortestSubarray(a, k),
    "SolutionStack": lambda a, k: SolutionStack().shortestSubarray(a, k),
    "SolutionDeque": lambda a, k: SolutionDeque().shortestSubarray(a, k),
    "SolutionBinary": lambda a, k: SolutionBinary().shortestSubarray(a, k),
    "Solution": lambda a, k: Solution().shortestSubarray(a, k),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([-5, -3, -1], 1, -1),                 # toàn âm
    ([10 ** 5], 10 ** 5, 1),
    ([5, -100, 5, 5], 10, 2),
    ([1] * 10, 10, 10),
    ([48, 99, 37, 4, -31], 140, 2),
]
for name, fn in methods.items():
    for a, k, exp in edge:
        assert fn(a, k) == exp, (name, a, k, fn(a, k), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force
for seed in range(1000):
    r_ = random.Random(seed)
    n = r_.randint(1, 15)
    a = [r_.randint(-10, 10) for _ in range(n)]
    k = r_.randint(1, 30)
    ref = shortest_brute(a, k)
    for name, fn in methods.items():
        assert fn(a, k) == ref, (name, a, k)
print("1000 bộ ngẫu nhiên: khớp brute force")

# 12.3 Dữ liệu vừa: khớp brute force
for seed in range(10):
    r_ = random.Random(5000 + seed)
    a = [r_.randint(-1000, 1000) for _ in range(800)]
    k = r_.randint(1, 20000)
    ref = shortest_brute(a, k)
    assert all(fn(a, k) == ref for name, fn in methods.items() if name != "brute")
print("n = 800: khớp brute force")

Ví dụ + edge cases: OK


1000 bộ ngẫu nhiên: khớp brute force


n = 800: khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng cửa sổ trượt như bài toàn số dương

**Code sai:**

In [22]:
def wrong_sliding_window(nums, k):
    n = len(nums)
    best, s, left = n + 1, 0, 0
    for right in range(n):
        s += nums[right]
        while s >= k:                               # SAI khi có số âm
            best = min(best, right - left + 1)
            s -= nums[left]
            left += 1
    return best if best <= n else -1

for a, k in [([84, -37, 32, 40, 95], 167), ([-28, 81, -20, 28, -29], 89), ([2, -1, 2], 3)]:
    print(a, k, "sai:", wrong_sliding_window(a, k), " đúng:", Solution().shortestSubarray(a, k))

[84, -37, 32, 40, 95] 167 sai: 5  đúng: 3
[-28, 81, -20, 28, -29] 89 sai: -1  đúng: 3
[2, -1, 2] 3 sai: 3  đúng: 3


**Vì sao sai:** với `[84, -37, 32, 40, 95]`, cửa sổ chỉ thu hẹp khi tổng ≥ `k`; khi bắt đầu từ 84 thì tổng `214` mới đủ, thu hẹp bỏ 84 → `130 < 167`, dừng — không bao giờ thử bỏ cả `-37` (bỏ phần tử âm làm tổng **tăng**!). **Code đúng:** deque đơn điệu trên tiền tố.

### Lỗi 2 — Đổi thứ tự hai vòng `while` (bỏ đuôi trước khi kiểm tra đầu)

**Code sai:**

In [23]:
def wrong_order(nums, k):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    dq = deque(); best = n + 1
    for j in range(n + 1):
        while dq and P[dq[-1]] >= P[j]:             # (2) trước
            dq.pop()
        dq.append(j)                                 # thêm j trước khi kiểm tra
        while dq and P[j] - P[dq[0]] >= k:          # (1) sau
            best = min(best, j - dq.popleft())
    return best if best <= n else -1

for a, k in [([1], 1), ([2, -1, 2], 3)]:
    print(a, k, "kết quả:", wrong_order(a, k), " đúng:", Solution().shortestSubarray(a, k))

[1] 1 kết quả: 1  đúng: 1
[2, -1, 2] 3 kết quả: 3  đúng: 3


Ở đây kết quả vẫn đúng trên các ví dụ (vì `P[j] − P[j] = 0 < k` nên `j` không tự ghép với chính nó), nhưng cách viết dễ gây nhầm khi biến thể có `k <= 0`. **Thói quen tốt:** kiểm tra đầu → bỏ đuôi → thêm `j`, đúng như chứng minh ở mục 10.

### Lỗi 3 — Bỏ đuôi với điều kiện `>` thay vì `>=` hoặc quên bỏ đầu

**Code sai:**

In [24]:
def wrong_no_popleft(nums, k):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    dq = deque(); best = n + 1
    for j in range(n + 1):
        for i in dq:                                 # SAI: duyệt cả deque thay vì bỏ đầu → O(n²)
            if P[j] - P[i] >= k:
                best = min(best, j - i)
        while dq and P[dq[-1]] >= P[j]:
            dq.pop()
        dq.append(j)
    return best if best <= n else -1

inc = list(range(1, 3001))                           # tăng dần: deque không bao giờ bị bỏ đuôi
t0 = time.perf_counter(); r1 = wrong_no_popleft(inc, 10 ** 9); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = Solution().shortestSubarray(inc, 10 ** 9); t2 = time.perf_counter() - t0
print(f"không bỏ đầu: {r1} ({t1 * 1000:.0f} ms) | đúng: {r2} ({t2 * 1000:.1f} ms)")

không bỏ đầu: -1 (180 ms) | đúng: -1 (0.6 ms)


**Vì sao sai:** kết quả đúng nhưng mất tính `O(n)` — mảng tăng dần giữ cả `n` phần tử trong deque → `O(n²)`. **Code đúng:** `popleft` sau khi ghi nhận (an toàn vì các `j` sau chỉ cho độ dài lớn hơn).

### Lỗi 4 — Trả `best` mà quên đổi "vô cực" thành `-1`

`best = n + 1` (hoặc `inf`) phải được đổi thành `-1` khi không tìm thấy. Ví dụ `[1, 2]`, `k = 4` → `-1`, không phải `3`.

In [25]:
print(Solution().shortestSubarray([1, 2], 4))

-1


## 14. Mẹo & nhận dạng pattern

- **"Mảng con có tổng ≥ / ≤ k" + có số âm** → **tổng tiền tố** + cấu trúc đơn điệu (deque/stack) hoặc heap; cửa sổ trượt chỉ dùng được khi mọi phần tử **không âm**.
- **Deque đơn điệu**: giữ ứng viên "không bị lấn át"; bỏ đầu khi đã dùng xong / hết hạn, bỏ đuôi khi bị phần tử mới tốt hơn mọi mặt.
- **"Ngắn nhất / dài nhất" + vị từ đơn điệu theo độ dài** → nhị phân theo đáp án.
- Bài liên quan: **209 Minimum Size Subarray Sum** (số dương), **239 Sliding Window Maximum**, **1499 Max Value of Equation**, **918 Maximum Sum Circular Subarray**, **325 Maximum Size Subarray Sum Equals k**.

## 15. Biến thể

1. **Mọi phần tử dương** (209): cửa sổ trượt `O(n)`.
2. **Dài nhất với tổng ≤ k** (có số âm): tiền tố + ngăn xếp đơn điệu giảm từ phải / nhị phân.
3. **Tổng đúng bằng k, dài nhất** (325): `dict` lưu vị trí đầu tiên của mỗi tiền tố.
4. **Mảng vòng**: nhân đôi mảng, giới hạn độ dài ≤ `n`, deque với điều kiện hết hạn `j − dq[0] > n`.
5. **Trả về chính mảng con**: lưu cặp `(i, j)` khi cập nhật `best`.

Minh hoạ biến thể 4 (mảng vòng) đối chiếu brute force:

In [26]:
def shortest_circular(nums, k):
    n = len(nums)
    a = nums + nums
    P = [0] + list(accumulate(a))
    dq = deque(); best = n + 1
    for j in range(2 * n + 1):
        while dq and j - dq[0] > n:                  # độ dài tối đa n
            dq.popleft()
        while dq and P[j] - P[dq[0]] >= k:
            best = min(best, j - dq.popleft())
        while dq and P[dq[-1]] >= P[j]:
            dq.pop()
        dq.append(j)
    return best if best <= n else -1

def shortest_circular_brute(nums, k):
    n = len(nums); best = n + 1
    for i in range(n):
        s = 0
        for L in range(1, n + 1):
            s += nums[(i + L - 1) % n]
            if s >= k:
                best = min(best, L); break
    return best if best <= n else -1

for seed in range(300):
    r_ = random.Random(seed)
    a = [r_.randint(-6, 6) for _ in range(r_.randint(1, 10))]
    k = r_.randint(1, 15)
    assert shortest_circular(a, k) == shortest_circular_brute(a, k), (a, k)
print("mảng vòng [5, -10, 3, 4], k=8:", shortest_circular([5, -10, 3, 4], 8))

mảng vòng [5, -10, 3, 4], k=8: 2


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi điểm đầu, cộng dần | `O(n²)` | `O(1)` | đối chiếu |
| 2. Heap tiền tố | min-heap, bỏ đỉnh khi dùng xong | `O(n log n)` | `O(n)` | dễ nghĩ, ít điều kiện |
| 3. Ngăn xếp + nhị phân | `P` tăng dần, tìm `i` lớn nhất | `O(n log n)` | `O(n)` | khi không bỏ đầu được (biến thể) |
| 4. Deque đơn điệu | bỏ đầu + bỏ đuôi | `O(n)` | `O(n)` | **nộp bài** |
| 5. Nhị phân theo đáp án | kiểm tra độ dài ≤ L bằng min trượt | `O(n log n)` | `O(n)` | khi vị từ đơn điệu dễ kiểm tra |

## 17. Bài tập

**Bài 1 (đoán output).** `nums = [3, -2, 5]`, `k = 4`? `k = 6`? `k = 7`?

**Bài 2 (tính tay).** Lập bảng trace deque cho `nums = [2, -1, 2]`, `k = 3`.

**Bài 3 (điền TODO).** Viết `shortest_with_indices(nums, k)` trả về `(độ dài, l, r)` của mảng con ngắn nhất (hoặc `None`).

**Bài 4 (tự viết lại).** Viết lời giải cho biến thể 1 (mọi phần tử dương) bằng cửa sổ trượt và chứng minh vì sao đúng khi không có số âm.

**Bài 5 (mini challenge).** Tìm độ dài **dài nhất** của mảng con có tổng **≤ k** (có số âm) trong `O(n log n)`.

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def shortest_with_indices_todo(nums, k):
    # TODO: như Solution; khi best giảm, lưu (i, j - 1)
    pass

# Bài 4 — TODO
def min_subarray_len_positive_todo(nums, k):
    # TODO: cửa sổ trượt: mở rộng right, khi tổng >= k thì thu hẹp left
    pass

# Bài 5 — TODO
def longest_sum_at_most_k_todo(nums, k):
    # TODO: cần i < j với P[i] >= P[j] - k, j - i lớn nhất; giữ "max tiền tố từ trái" tăng dần, nhị phân
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `P = [0, 3, 1, 6]`.
- Mức 2: `k = 4`: `5 >= 4` → 1; `k = 6`: `[3,-2,5]` = 6 → 3 (hoặc `[-2,5]` = 3 không đủ).
- Mức 3: `1`, `3`, `-1`.

**Bài 2.**
- Mức 1: `P = [0, 2, 1, 3]`.
- Mức 2: `j=2`: bỏ đuôi 1 (2 ≥ 1) → deque `[0, 2]`.
- Mức 3: `j=3`: `3 − P[0] = 3 ≥ 3` → bỏ đầu 0, độ dài 3; `3 − P[2] = 2 < 3` → dừng. Đáp án 3.

**Bài 3.**
- Mức 1: lưu thêm `(i, j)` khi cập nhật.
- Mức 2: mảng con là `nums[i:j]`, tức `l = i`, `r = j − 1`.
- Mức 3: kiểm tra `sum(nums[l:r+1]) >= k`.

**Bài 4.**
- Mức 1: khi mọi phần tử > 0, tổng cửa sổ tăng khi mở rộng, giảm khi thu hẹp.
- Mức 2: với mỗi `right`, `left` tối ưu chỉ tăng.
- Mức 3: so khớp với `Solution` trên dữ liệu dương.

**Bài 5.**
- Mức 1: cần `P[j] − P[i] <= k` ⇔ `P[i] >= P[j] − k`, tối đa `j − i`.
- Mức 2: với `j` cố định, muốn `i` **nhỏ nhất** có `P[i] >= P[j] − k`. Dãy `M[i] = max(P[0..i])` tăng dần → nhị phân trên `M`.
- Mức 3: `i` nhỏ nhất có `M[i] >= P[j] − k` cũng là `i` nhỏ nhất có `P[i] >= P[j] − k`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert [Solution().shortestSubarray([3, -2, 5], k) for k in (4, 6, 7)] == [1, 3, -1]

# Bài 2
assert Solution().shortestSubarray([2, -1, 2], 3) == 3

# Bài 3
def shortest_with_indices(nums, k):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    dq = deque(); best = None
    for j in range(n + 1):
        while dq and P[j] - P[dq[0]] >= k:
            i = dq.popleft()
            if best is None or j - i < best[0]:
                best = (j - i, i, j - 1)
        while dq and P[dq[-1]] >= P[j]:
            dq.pop()
        dq.append(j)
    return best

for seed in range(200):
    r_ = random.Random(seed)
    a = [r_.randint(-10, 10) for _ in range(r_.randint(1, 12))]; k = r_.randint(1, 25)
    res = shortest_with_indices(a, k)
    if res is None:
        assert Solution().shortestSubarray(a, k) == -1
    else:
        L, l, r = res
        assert L == r - l + 1 == Solution().shortestSubarray(a, k) and sum(a[l:r + 1]) >= k

# Bài 4
def min_subarray_len_positive(nums, k):
    n = len(nums); best = n + 1; s = left = 0
    for right, x in enumerate(nums):
        s += x
        while s >= k:
            best = min(best, right - left + 1)
            s -= nums[left]; left += 1
    return best if best <= n else -1

for seed in range(200):
    r_ = random.Random(seed)
    a = [r_.randint(1, 10) for _ in range(r_.randint(1, 15))]; k = r_.randint(1, 40)
    assert min_subarray_len_positive(a, k) == Solution().shortestSubarray(a, k)

# Bài 5
def longest_sum_at_most_k(nums, k):
    P = [0] + list(accumulate(nums))
    M = list(accumulate(P, max))                  # M[i] = max(P[0..i]) — tăng dần
    best = 0
    from bisect import bisect_left
    for j in range(1, len(P)):
        i = bisect_left(M, P[j] - k, 0, j)        # i nhỏ nhất trong [0, j) với M[i] >= P[j] - k
        if i < j:
            best = max(best, j - i)
    return best

def longest_sum_at_most_k_brute(nums, k):
    n = len(nums); best = 0
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            if s <= k:
                best = max(best, j - i + 1)
    return best

for seed in range(300):
    r_ = random.Random(seed)
    a = [r_.randint(-10, 10) for _ in range(r_.randint(1, 12))]; k = r_.randint(-10, 20)
    assert longest_sum_at_most_k(a, k) == longest_sum_at_most_k_brute(a, k), (a, k)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (deque đơn điệu trên tiền tố):**

```python
P = [0]*(n+1)
for i, x in enumerate(nums): P[i+1] = P[i] + x
dq = deque(); best = n + 1
for j in range(n + 1):
    while dq and P[j] - P[dq[0]] >= k: best = min(best, j - dq.popleft())   # dùng xong
    while dq and P[dq[-1]] >= P[j]: dq.pop()                                # bị lấn át
    dq.append(j)
return best if best <= n else -1
```

**Checklist trước khi submit:**

- [ ] Làm việc trên **tổng tiền tố** `P` độ dài `n + 1` (có `P[0] = 0`).
- [ ] Có số âm → **không** dùng cửa sổ trượt.
- [ ] Thứ tự: kiểm tra đầu → bỏ đuôi → thêm `j`.
- [ ] Bỏ đuôi với điều kiện `P[dq[-1]] >= P[j]`.
- [ ] Không tìm thấy → `-1`.